##  **INIT**

In [0]:

import requests
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from config.datasets import DATASETS

spark = SparkSession.builder.getOrCreate()




## **fetch function**

In [0]:
def fetch_records(ds, limit=1000):

    all_records = []
    offset = 0
    
    while True:
        params = {
            "resource_id": ds["resource_id"],
            "limit": limit,
            "offset": offset
        }
        response = requests.get(ds["api_url"], params=params)
        response.raise_for_status()
        
        data = response.json()
        records = data["result"]["records"]
        
        if not records:
            break
        
        all_records.extend(records)
        break
    
    return all_records

## **creat a dataframe from record**

In [0]:
def create_df_from_records(records):
    if not records:
        return None

    fields = set()
    for r in records:
        fields.update(r.keys())

    schema = StructType([
        StructField(f, StringType(), True)
        for f in fields
    ])

    return spark.createDataFrame(records, schema=schema)


## **write to delta table**

In [0]:
def write_delta(df, ds):
   
    full_table_name = f"{ds['catalog']}.{ds['schema']}.{ds['table']}"
    
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(full_table_name)
    
    print(f"Created Delta Table: {full_table_name}")


## **ingestion to bronze layer**

In [0]:
for ds in DATASETS:
    if not ds.get("enabled", True):
        print(f"Skipping {ds['name']} (disabled)")
        continue
    
    print(f"⬇Fetching {ds['name']}")
    records = fetch_records(ds)
    
    df = create_df_from_records(records)
    if df is None:
        print(f"No records for {ds['name']}")
        continue
    
    print(f"Writing {ds['name']} to Delta Table")
    write_delta(df, ds)